# Flowsheet Visualization

This notebook demonstrates the interactive visualization capabilities in difflow:

- Building flowsheet graphs programmatically
- Interactive visualization with zoom, pan, and tooltips
- Customizing node and edge styles
- Exporting to HTML for sharing

In [ ]:
import jax.numpy as jnp

from difflow import make_stream, get_flows
from difflow.visualization import (
    FlowsheetGraph,
    render_flowsheet,
    show_flowsheet,
    UNIT_STYLES,
)

## 1. Simple CSTR-Flash Example

Let's build and visualize a simple process with a reactor and flash separator.

In [ ]:
# Create the flowsheet graph
graph = FlowsheetGraph(name="CSTR-Flash Process")

# Add unit operations
graph.add_node("reactor", "CSTR-101", unit_type="CSTR", 
               params={"V": 10.0, "T": 350.0})
graph.add_node("flash", "Flash-101", unit_type="Flash",
               params={"T": 320.0, "P": 101325.0})

# Add feed and products
feed_stream = {"F_A": 100.0, "F_B": 0.0, "T": 300.0, "P": 101325.0}
graph.add_feed("reactor", "Fresh Feed", stream_data=feed_stream)
graph.add_product("flash", "Vapor Product", source_port="vapor")
graph.add_product("flash", "Liquid Product", source_port="liquid")

# Connect reactor to flash
reactor_outlet = {"F_A": 20.0, "F_B": 80.0, "T": 350.0, "P": 101325.0}
graph.add_edge("reactor", "flash", edge_id="S1", stream_data=reactor_outlet)

print(graph)

In [ ]:
# Render the flowsheet
fig = render_flowsheet(
    graph,
    layout="spring",  # Use spring layout (hierarchical 'dot' requires graphviz)
    title="CSTR-Flash Process",
    width=800,
    height=500,
)
fig.show()

**Interactive Features:**
- **Zoom**: Scroll or pinch to zoom in/out
- **Pan**: Drag to move around
- **Tooltips**: Hover over nodes and edges to see details
- **Toolbar**: Use the toolbar on the right for additional controls

## 2. Bio Manufacturing Process

Let's visualize a more complex mAb downstream process.

In [ ]:
# Create mAb process flowsheet
mab_graph = FlowsheetGraph(name="mAb Downstream Process")

# Upstream
mab_graph.add_node("bioreactor", "Fed-Batch\nBioreactor", 
                   unit_type="FedBatchBioreactor",
                   params={"V": 1000.0, "titer": "3 g/L"})

# Harvest
mab_graph.add_node("centrifuge", "Disc-Stack\nCentrifuge", 
                   unit_type="DiscStackCentrifuge",
                   params={"sigma": 5000.0, "rpm": 7000})

# Capture
mab_graph.add_node("proa", "Protein A\nCapture", 
                   unit_type="ProteinAChromatography",
                   params={"CV": 50.0, "q_max": 40.0})

# Polish
mab_graph.add_node("iex", "IEX\nPolish", 
                   unit_type="IonExchangeChromatography",
                   params={"mode": "flow-through"})

# Formulation
mab_graph.add_node("ufdf", "UF/DF\nFormulation", 
                   unit_type="TFF",
                   params={"MWCO": 30.0, "area": 10.0})

# Add feeds and products
mab_graph.add_feed("bioreactor", "Media + Glucose")
mab_graph.add_product("centrifuge", "Cell Paste", source_port="cells")
mab_graph.add_product("proa", "FT Waste", source_port="waste")
mab_graph.add_product("iex", "Bound Waste", source_port="waste")
mab_graph.add_product("ufdf", "Drug Substance")

# Add process streams
mab_graph.add_edge("bioreactor", "centrifuge", edge_id="Harvest",
                   stream_data={"F_mAb": 3000.0, "F_cells": 50000.0})
mab_graph.add_edge("centrifuge", "proa", edge_id="Clarified",
                   stream_data={"F_mAb": 2900.0, "F_HCP": 5000.0})
mab_graph.add_edge("proa", "iex", edge_id="ProA Pool",
                   stream_data={"F_mAb": 2700.0, "F_HCP": 50.0})
mab_graph.add_edge("iex", "ufdf", edge_id="IEX Pool",
                   stream_data={"F_mAb": 2600.0, "F_HCP": 5.0})

print(mab_graph)

In [ ]:
# Render the mAb process
fig_mab = render_flowsheet(
    mab_graph,
    layout="spring",
    title="mAb Downstream Process",
    width=1000,
    height=600,
    show_stream_labels=True,
    edge_width_scale=1.5,
)
fig_mab.show()

## 3. Available Unit Styles

The visualization module includes pre-defined styles for many unit operation types.

In [ ]:
# Show available unit styles
print("Available Unit Styles:")
print("=" * 50)
for name, style in sorted(UNIT_STYLES.items()):
    icon = style.icon or "  "
    print(f"  {icon} {name:30s} shape={style.shape}, color={style.color}")

## 4. Export to HTML

You can export the visualization to a standalone HTML file for sharing.

In [ ]:
from difflow.visualization.render import to_html

# Export to HTML file
html = to_html(mab_graph, filename="mab_process.html", layout="spring")
print("Saved to mab_process.html")
print(f"HTML size: {len(html) / 1024:.1f} KB")

## 5. Process with Recycle

Visualization handles recycle loops gracefully.

In [ ]:
# Create a process with recycle
recycle_graph = FlowsheetGraph(name="Process with Recycle")

# Units
recycle_graph.add_node("mixer", "Mixer", unit_type="Mixer")
recycle_graph.add_node("reactor", "Reactor", unit_type="CSTR", 
                       params={"conversion": 0.6})
recycle_graph.add_node("flash", "Flash Drum", unit_type="Flash")
recycle_graph.add_node("splitter", "Splitter", unit_type="Splitter")

# Feed and product
recycle_graph.add_feed("mixer", "Fresh Feed")
recycle_graph.add_product("flash", "Vapor", source_port="vapor")
recycle_graph.add_product("splitter", "Purge", source_port="purge")

# Main flow path
recycle_graph.add_edge("mixer", "reactor", edge_id="Combined Feed")
recycle_graph.add_edge("reactor", "flash", edge_id="Reactor Effluent")
recycle_graph.add_edge("flash", "splitter", edge_id="Liquid", 
                       source_port="liquid")

# Recycle stream
recycle_graph.add_edge("splitter", "mixer", edge_id="Recycle",
                       source_port="recycle",
                       stream_data={"F_A": 50.0, "F_B": 10.0})

# Render
fig_recycle = render_flowsheet(
    recycle_graph,
    layout="spring",
    title="Process with Recycle Loop",
    width=900,
    height=500,
)
fig_recycle.show()

## Summary

The visualization module provides:

| Feature | Description |
|---------|-------------|
| **Interactive** | Zoom, pan, tooltips via Plotly |
| **Graph-based** | Nodes = units, Edges = streams |
| **Styled** | Pre-defined styles for many unit types |
| **Exportable** | HTML, PNG, SVG output |
| **Informative** | Stream data shown in tooltips |

For best layouts, install graphviz and use `layout='dot'` for hierarchical process flows.